<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/07_gateway.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap07) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap07-mcp-gateway.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 6](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/06_contexto_y_coste.ipynb) · ➡️ [Cap. 8](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/08_rag_iceberg_s3vectors.ipynb)

# Capítulo 7 — Herramientas compartidas y gobernadas

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** TiendaNova lanza una app para repartidores que necesita consultar envíos. ¿Copiamos las herramientas en otro agente, con los datos y las credenciales dentro?

| | |
|---|---|
| **🧠 Componente agéntico** | Interoperabilidad de herramientas (MCP) |
| **La necesidad** | Publicar las herramientas **una vez**, con autenticación y límites, sobre los datos reales de la plataforma |
| **Lo que construimos** | **Gateway** (MCP) + Lambda sobre las tablas **Iceberg** (Athena) + autenticación M2M con **Identity** |
| **🏠 En local** | ✅ Funciona: un servidor MCP local por *stdio* (sección 0). Deja de bastar cuando otras apps tienen que llamarlo por Internet. |
| **☁️ Por qué AWS al salir a Internet** | Con las herramientas en el código, las credenciales viajan con cada agente y no hay límites: un bucle del agente puede tumbar el sistema de pedidos. El Gateway centraliza autenticación, *rate limits* y trazas, y en el capítulo 10 aplicará las políticas. |
| **En el ordenador sería…** | El **bus y los controladores de dispositivos**: un periférico se conecta una vez al sistema y cualquier programa lo usa a través de él, con permisos |
| **Requisitos del caso** | RF-01, RF-02; base de RF-03 y RNF-01 |
| **Prerrequisito · duración** | Capítulo 6 · 85 min |
| **⚠️ Dónde se rompe la analogía** | Un bus solo transporta; el Gateway además **autentica, limita y autoriza**: es bus y portero a la vez. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=7)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(7)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(7)
# ponerse_al_dia(7)   # ← descomenta si llegas tarde o algo se rompió

## 0. El concepto en local: un servidor MCP en tu máquina

Antes de publicar nada en Internet, veamos el protocolo. Este servidor expone dos herramientas de pedidos con **MCP**. El agente lo lanza como subproceso y habla con él por la entrada y salida estándar (*stdio*).

> ✋ **Predice antes de ejecutar.** El agente va a lanzar un servidor que no ha visto nunca. ¿Cómo sabrá qué herramientas tiene y con qué argumentos llamarlas?

In [ ]:
os.makedirs("mcp_local", exist_ok=True)

In [ ]:
%%writefile mcp_local/servidor_pedidos.py
"""Servidor MCP local (stdio) con herramientas de pedidos de TiendaNova. Capítulo 7, sección 0."""
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

DATOS = Path(__file__).resolve().parent.parent / "datos" / "tiendanova.json"
PEDIDOS = {p["pedido_id"]: p for p in json.loads(DATOS.read_text(encoding="utf-8"))["pedidos"]}
mcp = FastMCP("tiendanova-pedidos")


@mcp.tool()
def consultar_pedido(pedido_id: str) -> dict:
    """Devuelve estado, fechas, importes (en céntimos) y artículos de un pedido de TiendaNova."""
    p = PEDIDOS.get(pedido_id.upper())
    return {k: v for k, v in p.items() if k != "lineas_json"} if p else {"error": f"No existe el pedido {pedido_id}"}


@mcp.tool()
def consultar_envio(pedido_id: str) -> dict:
    """Devuelve transportista, estado y fechas estimada y real de entrega de un pedido."""
    p = PEDIDOS.get(pedido_id.upper())
    if not p:
        return {"error": f"No existe el pedido {pedido_id}"}
    return {k: p[k] for k in ("transportista", "estado", "eta", "fecha_entrega")}


if __name__ == "__main__":
    mcp.run()   # transporte stdio: el agente lanza este proceso y le habla por la entrada/salida estándar

In [ ]:
%%writefile mcp_local/cliente_local.py
"""Agente que descubre y usa las herramientas del servidor MCP local (stdio). Capítulo 7, sección 0."""
import os
import sys
from pathlib import Path

from mcp import StdioServerParameters, stdio_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from lab_utils import region, state_get  # noqa: E402

servidor = Path(__file__).resolve().parent / "servidor_pedidos.py"
servidor_local = MCPClient(lambda: stdio_client(StdioServerParameters(command=sys.executable, args=[str(servidor)])))
with servidor_local:
    herramientas = servidor_local.list_tools_sync()
    print("Herramientas descubiertas por MCP:", [t.tool_name for t in herramientas])
    agente = Agent(model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2),
                   tools=herramientas, callback_handler=None)
    r = agente(os.environ.get("PREGUNTA", "¿Dónde está mi pedido TN-1001?"))
    print("🤖", r)
    print("🔧", sorted(r.metrics.tool_metrics.keys()))

Lo ejecutamos como **proceso aparte** (igual que harías en una terminal con `python mcp_local/cliente_local.py`). Así funciona igual en Windows, macOS y Linux: dentro de Jupyter en Windows, el bucle de eventos del kernel no puede lanzar el servidor MCP como subproceso.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "mcp_local/cliente_local.py"], capture_output=True, text=True, encoding="utf-8")
print(r.stdout or r.stderr[-2000:])

El agente no sabe cómo están implementadas las herramientas: solo conoce el **protocolo**. Funciona… **en tu máquina**.

Ahora la app de repartidores, desde el móvil de cada repartidor, necesita llamar a estas herramientas **por Internet**:
- ¿Dónde se publican?
- ¿Quién puede llamarlas?
- ¿Qué pasa si un agente entra en bucle?
- ¿Con qué credenciales llegan a los datos de pedidos, que viven en la plataforma de datos de la empresa?

Eso es lo que resuelve el resto del capítulo: el mismo protocolo MCP, publicado en un **Gateway** con autenticación y límites, y con las herramientas leyendo los datos reales en **Iceberg**.

Las cuatro preguntas de arriba son la frontera entre tu máquina e Internet.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**El bus del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap07-mcp-gateway.html). Elige quién llama (Nova, la app de repartidores, un agente en bucle) y llama, en tu máquina y a través del Gateway. Vuelve cuando sepas responder a esta pregunta: **¿qué tres preguntas aparecen cuando una herramienta deja de vivir en tu máquina y pasa a llamarse por Internet, y quién las responde?**

**En dos líneas:** **MCP** es un estándar para exponer herramientas a cualquier agente: pregunta qué hay (`tools/list`) y llama (`tools/call`) sin conocer la implementación, así que la herramienta se publica una vez y la usan muchos agentes. En local el servidor es un subproceso por *stdio*; por Internet aparecen tres preguntas que el protocolo no responde (quién puede llamar, cuántas veces, con qué credenciales llega a los datos), y eso es lo que añade el Gateway: autenticación, límites, trazas y, en el capítulo 10, políticas.

> **Seguridad por defecto:** este capítulo crea el Gateway **con autenticación JWT desde el principio**, en lugar de usar `--authorizer-type NONE`, y con **protocolo MCP** (`--protocol-type MCP`: sin esa opción, la CLI 0.31 crea un Gateway HTTP, que no sirve herramientas MCP). El agente obtiene un token M2M (*client credentials*) a través de AgentCore Identity.

## 1. La Lambda con todas las herramientas, sobre Iceberg

La Lambda ya no tiene datos fijos: consulta las tablas **Iceberg** de S3 Tables con **Athena**.

- **Validación de entrada:** los identificadores que escribe el modelo se validan con una expresión regular antes de llegar a SQL, y las consultas usan parámetros.
- **Reglas de negocio:** el plazo de devolución, la excepción de higiene y el tope del reembolso al importe pagado las decide el código, no el modelo.

> **Latencia:** cada consulta de Athena tarda segundos. Es aceptable en el laboratorio, pero en producción las consultas puntuales de pedidos irían contra una base transaccional e Iceberg quedaría para histórico y analítica.

In [ ]:
os.makedirs("lambda", exist_ok=True)

In [ ]:
%%writefile lambda/nova_pedidos.py
"""Sistema de pedidos de TiendaNova sobre S3 Tables (Apache Iceberg), consultado con Athena.

Se publica como herramientas MCP a través de AgentCore Gateway.
"""
import json
import os
import re
import time
from datetime import date

import boto3

PLAZO_DEVOLUCION_DIAS = 30
PATRON_PEDIDO = re.compile(r"^TN-\d{4,6}$")
athena = boto3.client("athena")
CONTEXTO = {"Catalog": os.environ["ATHENA_CATALOG"], "Database": os.environ["ATHENA_DATABASE"]}
SALIDA = {"OutputLocation": os.environ["ATHENA_OUTPUT"]}


def _consulta(sql, params):
    qid = athena.start_query_execution(QueryString=sql, QueryExecutionContext=CONTEXTO,
                                       ResultConfiguration=SALIDA, ExecutionParameters=params)["QueryExecutionId"]
    while True:
        estado = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if estado["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(0.2)
    if estado["State"] != "SUCCEEDED":
        raise RuntimeError(estado.get("StateChangeReason", estado["State"]))
    filas = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    cols = [c.get("VarCharValue") for c in filas[0]["Data"]]
    return [dict(zip(cols, [c.get("VarCharValue") for c in f["Data"]])) for f in filas[1:]]


def _pedido(pedido_id):
    # El argumento lo escribe un modelo: se VALIDA antes de llegar a SQL.
    if not PATRON_PEDIDO.match(pedido_id):
        return None
    filas = _consulta("SELECT * FROM pedidos WHERE pedido_id = ?", [f"'{pedido_id}'"])
    if not filas:
        return None
    p = filas[0]
    p["lineas"] = json.loads(p.pop("lineas_json"))
    for campo in ("importe_bruto_centimos", "importe_pagado_centimos", "cupon_pct"):
        p[campo] = int(p[campo])
    return p


def lambda_handler(event, context):
    # El Gateway pasa los argumentos "planos" en el event y el nombre de la herramienta
    # en el client context, con formato Target___herramienta.
    herramienta = context.client_context.custom["bedrockAgentCoreToolName"].split("___")[-1]

    if herramienta == "escalar_a_humano":
        return {"ticket": "SUP-0001", "prioridad": event.get("prioridad", "normal"), "estado": "EN_COLA"}

    pedido_id = (event.get("pedido_id") or "").strip().upper()
    pedido = _pedido(pedido_id)
    if not pedido:
        return {"error": f"No existe el pedido {pedido_id} o el identificador no es válido"}

    if herramienta == "consultar_pedido":
        return pedido

    if herramienta == "consultar_envio":
        eventos = _consulta("SELECT ts, evento, detalle FROM envios_eventos WHERE pedido_id = ? ORDER BY ts DESC LIMIT 5",
                            [f"'{pedido_id}'"])
        return {k: pedido[k] for k in ("transportista", "estado", "eta", "fecha_entrega")} | {"ultimos_eventos": eventos}

    if herramienta == "solicitar_devolucion":
        # Reglas de negocio DETERMINISTAS: las decide el código, no el modelo.
        if pedido["estado"] != "ENTREGADO" or not pedido["fecha_entrega"]:
            return {"error": "Solo se pueden devolver pedidos entregados"}
        dias = (date.today() - date.fromisoformat(pedido["fecha_entrega"])).days
        if dias > PLAZO_DEVOLUCION_DIAS:
            return {"error": f"Fuera de plazo: {dias} días desde la entrega (máximo {PLAZO_DEVOLUCION_DIAS})"}
        avisos = [f"{l['nombre']}: no se admite si está desprecintado (higiene)"
                  for l in pedido["lineas"] if l.get("no_devolvible_si_desprecintado")]
        return {"rma": f"RMA-{pedido_id}", "estado": "ABIERTA", "motivo": event.get("motivo"), "avisos": avisos}

    if herramienta == "procesar_reembolso":
        importe = int(event["importe_centimos"])
        if importe > pedido["importe_pagado_centimos"]:
            return {"error": "El reembolso no puede superar el importe pagado del pedido"}
        return {"reembolso_id": f"RF-{pedido_id}", "importe_centimos": importe, "estado": "EMITIDO"}

    return {"error": f"Herramienta desconocida: {herramienta}"}

In [ ]:
import io, json, time, zipfile
import boto3

requiere("TABLE_BUCKET_ARN", "LAB_BUCKET", "ATHENA_CATALOG")
acct, r = state_get("ACCOUNT_ID"), region()
iam = boto3.client("iam")
lam = boto3.client("lambda", region_name=r)
ROLE = "nova-pedidos-role"
trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
         "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
try:
    role_arn = iam.create_role(RoleName=ROLE, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    iam.attach_role_policy(RoleName=ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE)["Role"]["Arn"]

tb, lab = state_get("TABLE_BUCKET_ARN"), state_get("LAB_BUCKET")
iam.put_role_policy(RoleName=ROLE, PolicyName="LecturaIcebergAthena", PolicyDocument=json.dumps({
    "Version": "2012-10-17", "Statement": [
        {"Effect": "Allow", "Action": ["athena:StartQueryExecution", "athena:GetQueryExecution", "athena:GetQueryResults"],
         "Resource": f"arn:aws:athena:{r}:{acct}:workgroup/primary"},
        {"Effect": "Allow", "Action": ["glue:GetCatalog", "glue:GetDatabase", "glue:GetTable", "glue:GetTables"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["s3tables:GetTableBucket", "s3tables:GetNamespace", "s3tables:GetTable",
                                       "s3tables:GetTableMetadataLocation", "s3tables:GetTableData"],
         "Resource": [tb, f"{tb}/*"]},
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket", "s3:GetBucketLocation",
                                       "s3:AbortMultipartUpload"],
         "Resource": [f"arn:aws:s3:::{lab}", f"arn:aws:s3:::{lab}/athena/*"]},
        {"Effect": "Allow", "Action": "lakeformation:GetDataAccess", "Resource": "*"},
    ]}))
time.sleep(10)   # propagación de IAM

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.write("lambda/nova_pedidos.py", "nova_pedidos.py")
entorno = {"Variables": {"ATHENA_CATALOG": state_get("ATHENA_CATALOG"), "ATHENA_DATABASE": ICEBERG_NAMESPACE,
                         "ATHENA_OUTPUT": state_get("ATHENA_OUTPUT")}}
for intento in range(6):
    try:
        try:
            fn = lam.create_function(FunctionName="nova-pedidos", Runtime="python3.13", Timeout=30,
                                     Handler="nova_pedidos.lambda_handler", Role=role_arn,
                                     Code={"ZipFile": buf.getvalue()}, Environment=entorno)
        except lam.exceptions.ResourceConflictException:
            fn = lam.update_function_code(FunctionName="nova-pedidos", ZipFile=buf.getvalue())
            lam.get_waiter("function_updated_v2").wait(FunctionName="nova-pedidos")
            lam.update_function_configuration(FunctionName="nova-pedidos", Environment=entorno, Timeout=30)
        break
    except lam.exceptions.InvalidParameterValueException:
        print("El rol aún no es asumible; reintentando…")
        time.sleep(10)
state_set(LAMBDA_ARN=fn["FunctionArn"])

**Prueba unitaria de la Lambda**, simulando el *client context* que envía el Gateway:

> ✋ **Predice antes de ejecutar.** Seis llamadas directas a la Lambda: ¿cuáles devolverán un error y por qué? Fíjate sobre todo en la última, con `TN-1001' OR '1'='1` como identificador.

In [ ]:
import base64

def probar_lambda(herramienta, **args):
    ctx = base64.b64encode(json.dumps({"custom": {"bedrockAgentCoreToolName": f"PedidosTarget___{herramienta}"}}).encode()).decode()
    lam.get_waiter("function_active_v2").wait(FunctionName="nova-pedidos")
    r = lam.invoke(FunctionName="nova-pedidos", ClientContext=ctx, Payload=json.dumps(args).encode())
    print(herramienta, "→", r["Payload"].read().decode())

probar_lambda("consultar_pedido", pedido_id="TN-1001")
probar_lambda("consultar_envio", pedido_id="TN-1005")                                  # incidencia de retraso
probar_lambda("solicitar_devolucion", pedido_id="TN-1002", motivo="no limpia bien")   # fuera de plazo
probar_lambda("solicitar_devolucion", pedido_id="TN-1004", motivo="no me gustan")     # aviso de higiene
probar_lambda("procesar_reembolso", pedido_id="TN-1001", importe_centimos=99999, motivo="test")   # supera el importe
probar_lambda("consultar_pedido", pedido_id="TN-1001' OR '1'='1")                    # rechazado por validación

## 2. Esquema de herramientas

In [ ]:
%%writefile NovaAssist/tools.json
[
  {
    "name": "consultar_pedido",
    "description": "Devuelve estado, fecha, importe en céntimos y artículos de un pedido de TiendaNova.",
    "inputSchema": {"type": "object",
                    "properties": {"pedido_id": {"type": "string", "description": "Identificador TN-NNNN"}},
                    "required": ["pedido_id"]}
  },
  {
    "name": "consultar_envio",
    "description": "Devuelve transportista, estado y fecha estimada de entrega de un pedido.",
    "inputSchema": {"type": "object",
                    "properties": {"pedido_id": {"type": "string", "description": "Identificador TN-NNNN"}},
                    "required": ["pedido_id"]}
  },
  {
    "name": "solicitar_devolucion",
    "description": "Abre una devolución (RMA) si el pedido está entregado y dentro del plazo de 30 días.",
    "inputSchema": {"type": "object",
                    "properties": {"pedido_id": {"type": "string"},
                                   "motivo": {"type": "string", "description": "Motivo indicado por el cliente"}},
                    "required": ["pedido_id", "motivo"]}
  },
  {
    "name": "procesar_reembolso",
    "description": "Emite un reembolso al método de pago original. Acción con impacto económico.",
    "inputSchema": {"type": "object",
                    "properties": {"pedido_id": {"type": "string"},
                                   "importe_centimos": {"type": "integer", "description": "Importe en céntimos"},
                                   "motivo": {"type": "string"}},
                    "required": ["pedido_id", "importe_centimos", "motivo"]}
  },
  {
    "name": "escalar_a_humano",
    "description": "Crea un ticket para el equipo humano de soporte con un resumen del caso.",
    "inputSchema": {"type": "object",
                    "properties": {"resumen": {"type": "string"},
                                   "prioridad": {"type": "string", "enum": ["baja", "normal", "alta"]}},
                    "required": ["resumen"]}
  }
]

## 3. Identidad de servicio (Cognito, *client credentials*)

In [ ]:
idp = boto3.client("cognito-idp", region_name=region())
if state_get("M2M_POOL_ID"):
    print("El pool M2M ya existe")
else:
    pool_id = idp.create_user_pool(PoolName="nova-m2m")["UserPool"]["Id"]
    dominio = f"nova-m2m-{state_get('ACCOUNT_ID')}"
    idp.create_user_pool_domain(Domain=dominio, UserPoolId=pool_id)
    idp.create_resource_server(UserPoolId=pool_id, Identifier="gateway", Name="Nova Gateway",
                               Scopes=[{"ScopeName": "invoke", "ScopeDescription": "Invocar herramientas"}])
    c = idp.create_user_pool_client(
        UserPoolId=pool_id, ClientName="nova-agent-m2m", GenerateSecret=True,
        AllowedOAuthFlows=["client_credentials"], AllowedOAuthScopes=["gateway/invoke"],
        AllowedOAuthFlowsUserPoolClient=True, SupportedIdentityProviders=["COGNITO"])["UserPoolClient"]
    state_set(M2M_POOL_ID=pool_id, M2M_DOMAIN=dominio, M2M_CLIENT_ID=c["ClientId"],
              M2M_CLIENT_SECRET=c["ClientSecret"], GATEWAY_SCOPES=["gateway/invoke"],
              M2M_DISCOVERY_URL=f"https://cognito-idp.{region()}.amazonaws.com/{pool_id}/.well-known/openid-configuration")

## 4. Gateway con `CUSTOM_JWT` y target Lambda

In [ ]:
s = state_load()
cli_args("add", "gateway", "--name", "NovaGateway", "--protocol-type", "MCP", "--authorizer-type", "CUSTOM_JWT",
         "--discovery-url", s["M2M_DISCOVERY_URL"], "--allowed-clients", s["M2M_CLIENT_ID"],
         "--client-id", s["M2M_CLIENT_ID"], "--client-secret", s["M2M_CLIENT_SECRET"])
cli_args("add", "gateway-target", "--name", "PedidosTarget", "--type", "lambda-function-arn",
         "--lambda-arn", s["LAMBDA_ARN"], "--tool-schema-file", "tools.json", "--gateway", "NovaGateway")
print(json.dumps(agentcore_json()["agentCoreGateways"], indent=2, ensure_ascii=False)[:1200])

### Despliegue

La CLI ha guardado el *client id* y el *secret* del pool M2M en `agentcore/.env.local` (no se versiona): con ellos, el despliegue crea el *credential provider* que usará el Runtime para pedir el token. Si en tu equipo el despliegue pidiera datos de forma interactiva, ejecútalo en una terminal (`cd labs/NovaAssist && agentcore deploy`).

In [ ]:
cli("agentcore deploy --yes")

In [ ]:
comprobar_gateway()   # para aquí con un mensaje claro si el despliegue no dejó el Gateway listo

## 5. Comprobar el Gateway

> ✋ **Predice antes de ejecutar.** `tools/list` sin token: ¿qué devolverá el Gateway? ¿Y con el token M2M?

In [ ]:
try:
    mcp_tools_list()                    # sin token → debe fallar (401)
except Exception as e:
    print("Sin token →", e)

herramientas_gw = mcp_tools_list(token=token_m2m())   # con token M2M → lista PedidosTarget___*

> Los nombres aparecen como `PedidosTarget___consultar_pedido`, con **tres guiones bajos**. Ese mismo nombre se usará en Cedar (capítulo 10). Si `token_m2m()` falla nada más crear el dominio de Cognito, espera un minuto y repite.

## 6. El agente consume el Gateway

In [ ]:
fijar_dependencia(VERSIONES["mcp"])
write_settings()

In [ ]:
%%writefile NovaAssist/app/NovaAgent/gateway.py
"""Cliente MCP del Gateway. El token M2M lo obtiene AgentCore Identity (credential provider de la CLI)."""
import asyncio
import os

from bedrock_agentcore.identity.auth import requires_access_token
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient

from settings import GATEWAY_CREDENTIAL_PROVIDER, GATEWAY_SCOPES, GATEWAY_URL


@requires_access_token(provider_name=GATEWAY_CREDENTIAL_PROVIDER, scopes=GATEWAY_SCOPES, auth_flow="M2M")
async def _token_identity(*, access_token: str) -> str:
    return access_token


def token_gateway() -> str:
    # Solo para pruebas locales desde el notebook se admite un token inyectado por variable de entorno.
    # En el Runtime esa variable no existe y el token se obtiene de AgentCore Identity.
    token = os.environ.get("NOVA_GATEWAY_TOKEN")
    return token if token else asyncio.run(_token_identity())


def cliente_gateway(token: str) -> MCPClient:
    return MCPClient(lambda: streamablehttp_client(GATEWAY_URL, headers={"Authorization": f"Bearer {token}"}))

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager

from contexto import VENTANA_MENSAJES, system_prompt
from gateway import cliente_gateway, token_gateway
from memoria import crear_session_manager, retrieval_config
from modelo import crear_modelo
from respuesta import respuesta

app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload, context):
    actor_id = payload.get("cliente_id", "anonimo")  # ⚠️ provisional: en el capítulo 9 saldrá del token
    gateway = cliente_gateway(token_gateway())
    with gateway, crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=crear_modelo(),
            system_prompt=system_prompt(payload),
            tools=gateway.list_tools_sync(),      # las herramientas ya no están en el código: las publica el Gateway
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=VENTANA_MENSAJES),
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    return respuesta(result)


if __name__ == "__main__":
    app.run()

In [ ]:
# Las herramientas y los datos locales ya no se usan: ahora viven en la Lambda y en Iceberg
import sys
for modulo in ("herramientas", "datos_pedidos"):
    (AGENT_DIR / f"{modulo}.py").unlink(missing_ok=True)
    sys.modules.pop(modulo, None)

## 7. Prueba local (token M2M inyectado solo para el notebook)

In [ ]:
os.environ["NOVA_GATEWAY_TOKEN"] = token_m2m()
probar_local("¿Dónde está mi pedido TN-1001?")
probar_local("Quiero devolver el robot del pedido TN-1002", cliente_id="C-002")   # la Lambda aplica el plazo
del os.environ["NOVA_GATEWAY_TOKEN"]

## 8. Desplegar y probar en el Runtime (el token llega de AgentCore Identity)

In [ ]:
cli('agentcore deploy --yes')

In [ ]:
invocar("¿Dónde está mi pedido TN-1001?")
invocar("Quiero devolver la cafetera del pedido TN-1003, no me convence")

> Si el Runtime no consigue el token (errores de *workload identity* o `AccessDenied`), comprueba que el nombre del *credential provider* de `settings.py` coincide con el de `agentcore.json` (`comprobar_gateway()` lo guarda) y vuelve a desplegar.

## Para pensar

- ¿Qué has ganado con el Gateway frente a las herramientas en el código?
- El token M2M identifica al **agente**, no al **cliente**. ¿Qué implica eso? (capítulo 9)
- Con 300 herramientas de 20 equipos, ¿qué pasaría con la ventana de contexto sin la búsqueda semántica?

## 🔎 Lo que acabamos de descubrir

Laura pregunta: *«¿Cuántos años de garantía tiene el robot aspirador?»*. No es un dato de ningún pedido ni está en la ROM del modelo: es **conocimiento de la empresa**, y vive en documentos.

In [ ]:
cuaderno()
taximetro_cerrar(7)
guardar_partida(7)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap07-mcp-gateway.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap07-mcp-gateway.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `08_rag_iceberg_s3vectors.ipynb` — *La biblioteca de la empresa*

---
**Siguiente:** ➡️ [Capítulo 8 · La biblioteca de la empresa](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/08_rag_iceberg_s3vectors.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap08). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.